# Practical 03: In-Memory Optimization via Dtype Downcasting & Deep Profiling

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to demonstrate in-memory optimization of large datasets. Profile baseline memory usage using memory_usage(deep=True), downcast integer columns (int64 -> int32/int16), downcast float columns (float64 -> float32), and convert repetitive string columns to category type. Report the total percentage reduction in RAM footprint.

In [1]:
import numpy as np
import pandas as pd

gen = np.random.default_rng(7)
n = 400_000
orders = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1, dtype="int64"),
    "customer_id": gen.integers(1, 45000, n),
    "store_id": gen.integers(1, 150, n),
    "quantity": gen.integers(1, 20, n),
    "amount": np.round(gen.gamma(2.0, 55.0, n), 2),
    "category": gen.choice(["Electronics", "Clothing", "Groceries", "Toys", "Books", "Beauty"], n).astype(object),
    "payment_method": gen.choice(["Credit Card", "Debit Card", "UPI", "Cash", "Net Banking"], n).astype(object),
    "city": gen.choice(["Delhi", "Jaipur", "Surat", "Indore", "Nagpur", "Pune", "Ahmedabad"], n).astype(object),
})
print(orders.head())

def mb(frame):
    """Total deep memory of a DataFrame in MB."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

baseline = mb(orders)
print(orders.dtypes, "\n")
print("Memory per column (MB):")
print((orders.memory_usage(deep=True) / 1024 ** 2).round(3))
print(f"\nBASELINE TOTAL: {baseline:.2f} MB")

opt = orders.copy()
for col in opt.select_dtypes(include="integer").columns:
    opt[col] = pd.to_numeric(opt[col], downcast="integer")

after_int = mb(opt)
print(opt.select_dtypes(include="integer").dtypes.to_dict())
print(f"After integer downcast: {after_int:.2f} MB")

for col in opt.select_dtypes(include="float").columns:
    opt[col] = pd.to_numeric(opt[col], downcast="float")

after_float = mb(opt)
print(opt.select_dtypes(include="float").dtypes.to_dict())
print(f"After float downcast: {after_float:.2f} MB")

for col in opt.select_dtypes(include=["object", "string"]).columns:
    if opt[col].nunique() / len(opt) < 0.3:
        opt[col] = opt[col].astype("category")

after_cat = mb(opt)
print(opt.dtypes)
print(f"\nAfter category conversion: {after_cat:.2f} MB")

reduction = (baseline - after_cat) / baseline * 100
print("=" * 45)
print(f"Baseline  : {baseline:8.2f} MB")
print(f"Optimised : {after_cat:8.2f} MB")
print(f"REDUCTION : {reduction:8.2f} %")
print("=" * 45)

,transaction_id,customer_id,store_id,quantity,amount,category,payment_method,city
0,1,42520,6,18,17.40,Groceries,Debit Card,Nagpur
1,2,28129,122,13,75.95,Toys,UPI,Jaipur
2,3,30788,112,9,14.43,Toys,Debit Card,Ahmedabad
3,4,40374,56,5,51.41,Beauty,Cash,Jaipur
4,5,26023,28,3,85.43,Beauty,Debit Card,Indore


transaction_id      int64
customer_id         int64
store_id            int64
quantity            int64
amount            float64
category           object
payment_method     object
city               object
dtype: object 

Memory per column (MB):
Index              0.000
transaction_id     3.052
customer_id        3.052
store_id           3.052
quantity           3.052
amount             3.052
category          21.425
payment_method    21.664
city              20.927
dtype: float64

BASELINE TOTAL: 79.27 MB
{'transaction_id': dtype('int32'), 'customer_id': dtype('int32'), 'store_id': dtype('int16'), 'quantity': dtype('int8')}
After integer downcast: 71.26 MB
{'amount': dtype('float32')}
After float downcast: 69.74 MB
transaction_id       int32
customer_id          int32
store_id             int16
quantity              int8
amount             float32
category          category
payment_method    category
city              category
dtype: object

After category conversion: 6.87 MB
Baseli